# Machine Learning - Klasifikasi Penyakit Ginjal Kronik menggunakan WKNN
## Weighted K-Nearest Neighbor (WKNN) Classification untuk Dataset Chronic Kidney Disease

Dataset ini berisi data medis pasien dengan berbagai parameter kesehatan untuk memprediksi status penyakit ginjal kronik.

**Dataset**: `dataset/penyakit_ginjal_kronik.csv` (400 pasien, 24 fitur klinis)

**Target**:
- target = 0 → Tidak terdeteksi penyakit ginjal kronik (notckd)
- target = 1 → Terdeteksi penyakit ginjal kronik (ckd)

**Base method**: **W-KNN (Weighted K-Nearest Neighbors)** dengan validasi cross-validation 5-fold

## 🗺️ Pipeline Machine Learning

```
Pengumpulan Data
        ↓
Eksplorasi Data (EDA)
        ↓
Preprocessing & Cleaning
        ↓
Scaling Data (MinMaxScaler)
        ↓
Split Data (80/20)
        ↓
Training Model WKNN
        ↓
Evaluasi Model
        ↓
Hyperparameter Tuning
        ↓
Visualisasi Hasil
```

## 1. Load dan Explore Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

import warnings
warnings.filterwarnings('ignore')

# Konfigurasi plotting
plt.style.use('default')
sns.set_palette('husl')

## 1. Load dan Explore Dataset

In [ ]:
# Load dataset
file_path = 'dataset/penyakit_ginjal_kronik.csv'

# Membaca dataset
df = pd.read_csv(file_path)

# Membersihkan karakter ? menjadi NaN
# Dataset CKD sering memiliki nilai seperti '?', '\t?' dan spasi tersembunyi
df = df.replace(r'^\s*\?\s*$', np.nan, regex=True)

# Membersihkan spasi/tab tersembunyi
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str).str.strip()

print('='*60)
print('INFORMASI DATASET AWAL')
print('='*60)

print(f'Ukuran Dataset: {df.shape[0]} baris × {df.shape[1]} kolom')

print('\nTipe Data:')
print(df.dtypes)

print('\nMissing Values:')
print(df.isnull().sum())

print('\nStatistik Deskriptif:')
print(df.describe())

print('\nLima Baris Pertama Dataset:')
print(df.head())

In [ ]:
## 2. Visualisasi Distribusi Target

print('\nDistribusi Kelas Target:')
print(df['klasifikasi'].value_counts())

print('\nPersentase Kelas:')
print(df['klasifikasi'].value_counts(normalize=True) * 100)

fig, axes = plt.subplots(1, 2, figsize=(14,5))

# Bar plot
sns.countplot(x='klasifikasi', data=df, ax=axes[0])
axes[0].set_title('Distribusi Kelas Penyakit Ginjal Kronik')
axes[0].set_xlabel('Kelas')
axes[0].set_ylabel('Jumlah')

# Pie chart
class_counts = df['klasifikasi'].value_counts()
axes[1].pie(class_counts.values,
            labels=class_counts.index,
            autopct='%1.1f%%',
            startangle=90)
axes[1].set_title('Persentase Kelas')

plt.tight_layout()
plt.show()

## 3. Data Preprocessing dan Cleaning

In [ ]:
# Menghapus kolom ID jika ada
if 'id' in df.columns:
    df.drop('id', axis=1, inplace=True)

print('Kolom ID berhasil dihapus')

# Cek duplikat
print('\nJumlah Data Duplikat:', df.duplicated().sum())

if df.duplicated().sum() > 0:
    df = df.drop_duplicates()
    print('Data duplikat berhasil dihapus')

print('Ukuran Dataset Setelah Cleaning:', df.shape)

## 3.2 Konversi Kolom Numerik dan Imputasi

# Konversi kolom yang seharusnya numerik
numeric_candidate = [
    'umur',
    'tekanandarah',
    'gravitas',
    'albumin',
    'sugar',
    'gds',
    'ureum',
    'kreatinin',
    'natrium',
    'kalium',
    'hemoglobin',
    'mcv',
    'seldarahputih',
    'seldarahmerah'
]

for col in numeric_candidate:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns

print('='*60)
print('IDENTIFIKASI KOLOM NUMERIK DAN KATEGORIKAL')
print('='*60)

print('\nKolom Numerik:')
print(list(numerical_cols))

print('\nKolom Kategorikal:')
print(list(categorical_cols))

## 3.3 Imputasi Missing Value

In [ ]:
print('='*60)
print('PENANGANAN MISSING VALUE')
print('='*60)

print('\nJumlah Missing Value Sebelum Imputasi:')
print(df.isnull().sum())

# Imputasi numerik dengan mean
num_imputer = SimpleImputer(strategy='mean')
df[numerical_cols] = num_imputer.fit_transform(df[numerical_cols])

# Imputasi kategorikal dengan modus (most_frequent)
cat_imputer = SimpleImputer(strategy='most_frequent')
df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])

print('\nJumlah Missing Value Setelah Imputasi:')
print(df.isnull().sum().sum())

## 3.4 Encoding Data Kategorikal

In [ ]:
mapping = {
    'normal': 1,
    'abnormal': 0,
    'present': 1,
    'notpresent': 0,
    'yes': 1,
    'no': 0,
    'good': 1,
    'poor': 0,
    'ckd': 1,
    'notckd': 0
}

# Terapkan encoding
df.replace(mapping, inplace=True)

# Memastikan data sudah numerik
print('='*60)
print('TIPE DATA SETELAH ENCODING')
print('='*60)

print('\nTipe Data:')
print(df.dtypes)

# Konversi ulang seluruh kolom jika memungkinkan
for col in df.columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except:
        pass

print('\nTipe Data Final:')
print(df.dtypes)

## 4. Deteksi Outlier Menggunakan IQR

In [ ]:
print('='*60)
print('DETEKSI OUTLIER MENGGUNAKAN IQR')
print('='*60)

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

    print(f'{col}: {len(outliers)} outlier')

## 5. Feature Scaling

WKNN adalah algoritma berbasis jarak sehingga sangat sensitif terhadap skala data.

Menggunakan MinMaxScaler.

In [ ]:
# Pisahkan fitur dan target
X = df.drop('klasifikasi', axis=1)
y = df['klasifikasi']

print('='*60)
print('FEATURE SCALING MENGGUNAKAN MINMAXSCALER')
print('='*60)

print('\nJumlah fitur:', X.shape[1])
print('Jumlah data:', X.shape[0])

# Pastikan seluruh fitur numerik
print('Tipe data fitur:')
print(X.dtypes)

# Scaling
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

print('\nData berhasil dinormalisasi')

## 6. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('='*60)
print('TRAIN TEST SPLIT')
print('='*60)

print('Jumlah Data Training:', X_train.shape[0])
print('Jumlah Data Testing:', X_test.shape[0])

## 7. Implementasi Weighted K-Nearest Neighbor (WKNN)

In [ ]:
from sklearn.metrics.pairwise import euclidean_distances

class WKNN:
    def __init__(self, k=5):
        self.k = k

    def fit(self, X_train, y_train):
        self.X_train = X_train
        self.y_train = np.array(y_train)

    def predict(self, X_test):
        predictions = []

        for test_point in X_test:
            # Hitung jarak euclidean
            distances = np.sqrt(np.sum((self.X_train - test_point)**2, axis=1))

            # Ambil indeks K tetangga terdekat
            k_indices = np.argsort(distances)[:self.k]

            # Ambil label dan jarak
            k_labels = self.y_train[k_indices]
            k_distances = distances[k_indices]

            # Hitung weight
            weights = 1 / (k_distances + 1e-5)

            # Voting berbobot
            class_weights = {}

            for label, weight in zip(k_labels, weights):
                if label not in class_weights:
                    class_weights[label] = 0

                class_weights[label] += weight

            # Ambil kelas dengan weight terbesar
            prediction = max(class_weights, key=class_weights.get)

            predictions.append(prediction)

        return np.array(predictions)

## 8. Training Model WKNN

In [ ]:
print('='*60)
print('TRAINING MODEL WKNN')
print('='*60)

# Membuat model
wknn = WKNN(k=5)

# Training
wknn.fit(X_train, y_train)

# Prediksi
y_pred = wknn.predict(X_test)

print('\nPrediksi berhasil dilakukan')

In [ ]:
## 9. Evaluasi Model

from sklearn.metrics import roc_auc_score

print('='*60)
print('EVALUASI MODEL WKNN')
print('='*60)

# Hitung berbagai metrik untuk Testing Set
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f'  Metrik Performa pada Testing Set:')
print(f'  Accuracy:  {accuracy:.4f}')
print(f'  Precision: {precision:.4f}')
print(f'  Recall:    {recall:.4f}')
print(f'  F1-Score:  {f1:.4f}')

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

print(f'\nConfusion Matrix:')
print(cm)

# Classification Report
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred))

# ROC-AUC Score
y_scores = y_pred
roc_auc = roc_auc_score(y_test, y_scores)

print(f'ROC-AUC Score: {roc_auc:.4f}')

# ROC Curve
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_test, y_scores)

print('✓ Evaluasi model selesai!')

# Data untuk grafik performance metrics
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

metric_values = [
    accuracy,
    precision,
    recall,
    f1
]

In [ ]:
## 10. Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import KFold

print('='*60)
print('HYPERPARAMETER TUNING WKNN')
print('='*60)

k_range = range(1, 21)

best_k = 1
best_accuracy = 0

accuracy_list = []

for k in k_range:

    cv_scores = []

    kf = KFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    for train_index, test_index in kf.split(X_scaled):

        X_train_cv = X_scaled[train_index]
        X_test_cv = X_scaled[test_index]

        y_train_cv = y.iloc[train_index]
        y_test_cv = y.iloc[test_index]

        model = WKNN(k=k)

        model.fit(X_train_cv, y_train_cv)

        pred = model.predict(X_test_cv)

        score = accuracy_score(y_test_cv, pred)

        cv_scores.append(score)

    mean_accuracy = np.mean(cv_scores)

    accuracy_list.append(mean_accuracy)

    print(f'K={k} -> Accuracy={mean_accuracy:.4f}')

    if mean_accuracy > best_accuracy:
        best_accuracy = mean_accuracy
        best_k = k

print('\nBest K:', best_k)
print('Best Accuracy:', round(best_accuracy, 4))

plt.figure(figsize=(10,5))

plt.plot(k_range, accuracy_list, marker='o')

plt.title('Hyperparameter Tuning WKNN')
plt.xlabel('Nilai K')
plt.ylabel('Cross Validation Accuracy')

plt.grid(True)
plt.show()

## 11. Visualisasi Hasil

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16,12))

# 1 Accuracy vs K
axes[0,0].plot(k_range, accuracy_list, marker='o')
axes[0,0].set_title('Accuracy vs K')
axes[0,0].set_xlabel('Nilai K')
axes[0,0].set_ylabel('Accuracy')
axes[0,0].grid(True)

# 2 Confusion Matrix
sns.heatmap(cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=['Non CKD', 'CKD'],
            yticklabels=['Non CKD', 'CKD'],
            ax=axes[0,1])

axes[0,1].set_title('Confusion Matrix')
axes[0,1].set_xlabel('Predicted')
axes[0,1].set_ylabel('Actual')

# 3 ROC Curve
axes[1,0].plot(fpr, tpr, linewidth=2,
               label=f'AUC={roc_auc:.4f}')
axes[1,0].plot([0,1], [0,1], linestyle='--')
axes[1,0].set_title('ROC Curve')
axes[1,0].set_xlabel('False Positive Rate')
axes[1,0].set_ylabel('True Positive Rate')
axes[1,0].legend()
axes[1,0].grid(True)

# 4 Performance Metrics
bars = axes[1,1].bar(metric_names, metric_values)
axes[1,1].set_title('Performance Metrics')
axes[1,1].set_ylim(0, 1.1)
axes[1,1].grid(axis='y')

for bar in bars:
    height = bar.get_height()

    axes[1,1].text(
        bar.get_x() + bar.get_width()/2,
        height + 0.02,
        f'{height:.4f}',
        ha='center'
    )

plt.tight_layout()
plt.show()

## 12. Kesimpulan dan Analisis

In [ ]:
print('='*70)
print(' '*15 + 'RINGKASAN HASIL MACHINE LEARNING WKNN')
print('='*70)

print(f'📊 DATASET INFORMATION:')
print(f'   • Total sampel: {len(df)}')
print(f'   • Total fitur: {X.shape[1]}')
print(f'   • Kelas target: 2 (CKD dan Non-CKD)')
print(f'   • Training samples: {X_train.shape[0]} ({X_train.shape[0]/len(df)*100:.1f}%)')
print(f'   • Testing samples: {X_test.shape[0]} ({X_test.shape[0]/len(df)*100:.1f}%)')

print(f'\n🔧 PREPROCESSING:')
print(f'   • Missing value ditangani: Mean & Most Frequent Imputation')
print(f'   • Karakter ? dan tab dibersihkan')
print(f'   • Duplikat dihapus: {len(df) - df.drop_duplicates().shape[0]} baris')
print(f'   • Outlier dianalisis: Menggunakan metode IQR')
print(f'   • Feature scaling: MinMaxScaler (0-1 normalization)')

print(f'\n🏆 MODEL OPTIMAL:')
print(f'   • Algorithm: Weighted K-Nearest Neighbor (WKNN)')
print(f'   • Nilai K optimal: {best_k}')
print(f'   • Accuracy: {best_accuracy:.4f} ({best_accuracy*100:.2f}%)')
print(f'   • Precision: {precision:.4f}')
print(f'   • Recall: {recall:.4f}')
print(f'   • F1-Score: {f1:.4f}')
print(f'   • ROC-AUC Score: {roc_auc:.4f}')

print(f'\n📈 ANALISIS MODEL:')
print(f'   Model WKNN berhasil melakukan klasifikasi penyakit')
print(f'   ginjal kronik dengan performa yang sangat baik.')
print(f'   Penggunaan pembobotan jarak pada WKNN membuat')
print(f'   tetangga terdekat memiliki pengaruh lebih besar')
print(f'   dibanding data yang jaraknya lebih jauh.')
print(f'   Hal ini meningkatkan kemampuan model dalam')
print(f'   mengenali pola medis pasien CKD secara lebih akurat.')

print(f'\n✅ KESIMPULAN:')
print(f'   Model WKNN dengan k={best_k} memberikan performa terbaik')
print(f'   dengan akurasi {best_accuracy*100:.2f}% pada testing set.')
print(f'   Model ini mampu memprediksi status penyakit ginjal')
print(f'   kronik dengan baik berdasarkan parameter medis pasien.')
print(f'   Hasil penelitian menunjukkan bahwa metode WKNN')
print(f'   sangat cocok digunakan untuk klasifikasi dataset medis')
print(f'   khususnya penyakit ginjal kronik.')

print('\n' + '='*70)
print('✓ ANALISIS SELESAI')
print('='*70)

In [ ]:
def evaluate(model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    try:
        y_proba = model.predict_proba(X_te)[:, 1]
        roc = roc_auc_score(y_te, y_proba)
    except Exception:
        roc = None
    return {
        'accuracy': accuracy_score(y_te, y_pred),
        'precision': precision_score(y_te, y_pred, zero_division=0),
        'recall': recall_score(y_te, y_pred, zero_division=0),
        'f1': f1_score(y_te, y_pred, zero_division=0),
        'roc_auc': roc,
    }


def _safe_resample(resampler, X_tr, y_tr):
    if resampler is None:
        return X_tr, y_tr.copy(), False
    try:
        X_r, y_r = clone(resampler).fit_resample(X_tr, y_tr)
        return X_r, y_r, False
    except (ValueError, RuntimeError) as exc:
        print(f"   [WARN] Resampler gagal -> fallback ke data asli: {exc}")
        return X_tr, y_tr.copy(), True


results = []
for sc_name, scaler in SCALERS.items():
    X_tr_s = scaler.fit_transform(X_train)
    X_te_s = scaler.transform(X_test)
    for rs_name, resampler in RESAMPLERS.items():
        X_tr_r, y_tr_r, fallback = _safe_resample(resampler, X_tr_s, y_train)
        for mdl_name, mdl in MODELS.items():
            m = clone(mdl)
            metrics = evaluate(m, X_tr_r, X_te_s, y_tr_r, y_test)
            results.append({'Scaler': sc_name, 'Resampler': rs_name,
                            'Model': mdl_name, 'fallback_no_resample': fallback,
                            **metrics})
            tag = ' (fallback)' if fallback else ''
            print(f"[{sc_name:>14} | {rs_name:>18}{tag:<11} | {mdl_name:>20}] "
                  f"F1={metrics['f1']:.4f}  AUC={metrics['roc_auc']:.4f}")

results_df = pd.DataFrame(results)
print('\nTotal hasil:', len(results_df))
results_df.head()


## STEP 8 - Ringkasan Eksperimen

> 🏷️ **Fase Pipeline:** 📊 Evaluasi Model

Top kombinasi global biasanya didominasi 1 model. Untuk fairness, kita
juga tampilkan **best per model** dan pivot Model × Scaler / Resampler.


In [ ]:
results_sorted = results_df.sort_values('f1', ascending=False).reset_index(drop=True)
print(f'Total kombinasi: {len(results_sorted)}')
print('\n=== Top-10 kombinasi global ===')
results_sorted.head(10).round(4)


In [ ]:
print('Distribusi model di Top-10:')
print(results_sorted.head(10)['Model'].value_counts().to_string())
print('\nDistribusi model di Top-20:')
print(results_sorted.head(20)['Model'].value_counts().to_string())


In [ ]:
best_per_model = (results_sorted
                  .sort_values('f1', ascending=False)
                  .groupby('Model', as_index=False)
                  .first()
                  .sort_values('f1', ascending=False)
                  .reset_index(drop=True))
print('=== Konfigurasi terbaik per model ===')
best_per_model[['Model', 'Scaler', 'Resampler',
                'accuracy', 'precision', 'recall', 'f1', 'roc_auc']].round(4)


In [ ]:
pivot_scaler = (results_df.pivot_table(index='Model', columns='Scaler',
                                       values='f1', aggfunc='mean').round(4))
pivot_resampler = (results_df.pivot_table(index='Model', columns='Resampler',
                                          values='f1', aggfunc='mean').round(4))
print('=== Rata-rata F1 per (Model, Scaler) ===')
print(pivot_scaler)
print('\n=== Rata-rata F1 per (Model, Resampler) ===')
print(pivot_resampler)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.heatmap(pivot_scaler, annot=True, fmt='.4f', cmap='YlGnBu', ax=axes[0])
axes[0].set_title('Rata-rata F1 per Model × Scaler')
sns.heatmap(pivot_resampler, annot=True, fmt='.4f', cmap='YlOrRd', ax=axes[1])
axes[1].set_title('Rata-rata F1 per Model × Resampler')
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=best_per_model, x='Model', y='f1', palette='viridis', ax=axes[0])
axes[0].set_title('F1-Score Terbaik per Model')
axes[0].set_ylim(0, 1.05)
for i, row in best_per_model.iterrows():
    axes[0].text(i, row['f1'] + 0.01, f"{row['f1']:.3f}", ha='center')

sns.barplot(data=best_per_model, x='Model', y='roc_auc', palette='magma', ax=axes[1])
axes[1].set_title('ROC AUC Terbaik per Model')
axes[1].set_ylim(0, 1.05)
for i, row in best_per_model.iterrows():
    axes[1].text(i, row['roc_auc'] + 0.01, f"{row['roc_auc']:.3f}", ha='center')

plt.tight_layout()
plt.show()


## STEP 9 - Pemilihan Model: W-KNN (sesuai instruksi)

> 🏷️ **Fase Pipeline:** 📊 Evaluasi Model (Cross Validation)

Validasi performa W-KNN dengan **5-fold cross validation** dan bandingkan
dengan kandidat lain agar pemilihan tetap empiris.


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_summary = []

for _, row in best_per_model.iterrows():
    sc = SCALERS[row['Scaler']]
    rsp = RESAMPLERS[row['Resampler']]
    mdl = clone(MODELS[row['Model']])

    Xtr_s = sc.fit_transform(X_train)
    Xtr_r, ytr_r, _ = _safe_resample(rsp, Xtr_s, y_train)

    cv = cross_val_score(mdl, Xtr_r, ytr_r, cv=skf, scoring='f1', n_jobs=-1)
    cv_summary.append({
        'Model': row['Model'],
        'Scaler': row['Scaler'],
        'Resampler': row['Resampler'],
        'cv_f1_mean': cv.mean(),
        'cv_f1_std': cv.std(),
        'test_f1': row['f1'],
        'test_roc_auc': row['roc_auc'],
    })

cv_df = pd.DataFrame(cv_summary).sort_values('cv_f1_mean', ascending=False).reset_index(drop=True)
cv_df.round(4)


In [ ]:
# Pilih W-KNN sebagai base method (sesuai instruksi tugas)
wknn_row = cv_df[cv_df['Model'] == 'W-KNN'].iloc[0]
BEST = wknn_row.copy()

print(f"Base method (sesuai instruksi tugas): {BEST['Model']}")
print(f"  Konfigurasi   : Scaler={BEST['Scaler']}, Resampler={BEST['Resampler']}")
print(f"  CV F1 mean    : {BEST['cv_f1_mean']:.4f}  (std: {BEST['cv_f1_std']:.4f})")
print(f"  Test F1       : {BEST['test_f1']:.4f}")
print(f"  Test ROC AUC  : {BEST['test_roc_auc']:.4f}")

print('\nRanking semua model:')
print(cv_df.round(4).to_string(index=False))


## STEP 10 - Hyperparameter Tuning W-KNN

> 🏷️ **Fase Pipeline:** 🎛️ Tuning Model

Tuning W-KNN dengan `RandomizedSearchCV`. Parameter penting:
- `n_neighbors` — jumlah tetangga
- `weights` — `'uniform'` vs `'distance'` (W-KNN pakai distance)
- `metric` — `'euclidean'`, `'manhattan'`, `'minkowski'`
- `p` — exponent Minkowski


In [ ]:
PARAM_GRID_KNN = {
    'n_neighbors': [3, 5, 7, 9, 11, 13, 15],
    'weights': ['distance'],  # W-KNN, fix
    'metric': ['euclidean', 'manhattan', 'minkowski'],
    'p': [1, 2, 3],
}

best_scaler = SCALERS[BEST['Scaler']]
best_resampler = RESAMPLERS[BEST['Resampler']]
Xtr_s = best_scaler.fit_transform(X_train)
Xte_s = best_scaler.transform(X_test)
Xtr_final, ytr_final, _ = _safe_resample(best_resampler, Xtr_s, y_train)

base = clone(MODELS['W-KNN'])
search = RandomizedSearchCV(
    base, param_distributions=PARAM_GRID_KNN,
    n_iter=15, cv=5, scoring='f1', n_jobs=-1,
    random_state=RANDOM_STATE, verbose=1,
)
search.fit(Xtr_final, ytr_final)

print('Best params  :', search.best_params_)
print(f"Best CV F1   : {search.best_score_:.4f}")


In [ ]:
def metrik(m):
    y_pred = m.predict(Xte_s)
    y_proba = m.predict_proba(Xte_s)[:, 1]
    return {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_proba),
    }

default_model = clone(MODELS['W-KNN']).fit(Xtr_final, ytr_final)
tuned_model = search.best_estimator_

compare_df = pd.DataFrame([
    {'Model': 'W-KNN (default)', **metrik(default_model)},
    {'Model': 'W-KNN (tuned)', **metrik(tuned_model)},
])
compare_df.round(4)


## STEP 11 - Analisis Mendalam W-KNN (setelah tuning)

> 🏷️ **Fase Pipeline:** 📊 Evaluasi Model (analisis mendalam)


In [ ]:
best_model = tuned_model
y_pred = best_model.predict(Xte_s)
y_proba = best_model.predict_proba(Xte_s)[:, 1]

print('Classification Report (W-KNN tuned):')
print(classification_report(y_test, y_pred, target_names=['Tidak CKD', 'CKD']))
print('Best params:', search.best_params_)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Tidak CKD', 'CKD'],
            yticklabels=['Tidak CKD', 'CKD'], ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix - W-KNN (tuned)')

fpr, tpr, _ = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)
axes[1].plot(fpr, tpr, label=f"AUC = {auc:.4f}", linewidth=2)
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve - W-KNN (tuned)')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()


**Catatan:** W-KNN tidak punya `feature_importances_` bawaan
(non-parametric, lazy learner). Untuk menilai kontribusi fitur, gunakan
proxy seperti **permutation importance** atau lihat korelasi dari Step 5.


In [ ]:
# Permutation importance sebagai proxy feature importance untuk W-KNN
from sklearn.inspection import permutation_importance

perm = permutation_importance(best_model, Xte_s, y_test,
                              n_repeats=10, random_state=RANDOM_STATE,
                              scoring='f1')
perm_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std,
}).sort_values('importance_mean', ascending=True)

plt.figure(figsize=(8, 7))
plt.barh(perm_df['feature'], perm_df['importance_mean'],
         xerr=perm_df['importance_std'], color='teal')
plt.title('Permutation Importance - W-KNN (tuned)')
plt.xlabel('Importance (drop in F1 ketika fitur dipermutasi)')
plt.tight_layout()
plt.show()
print(perm_df.sort_values('importance_mean', ascending=False))


## STEP 12 - Perbandingan Head-to-Head Semua Kandidat

> 🏷️ **Fase Pipeline:** 📊 Evaluasi Model (head-to-head perbandingan)

Kita konsolidasikan performa semua model dengan konfigurasi terbaiknya
+ versi tuned W-KNN.


In [ ]:
import time

def time_model(mdl, sc_name, rs_name):
    sc = SCALERS[sc_name]
    rs = RESAMPLERS[rs_name]
    Xtr_s = sc.fit_transform(X_train)
    Xte_s = sc.transform(X_test)
    Xtr_r, ytr_r, _ = _safe_resample(rs, Xtr_s, y_train)

    m = clone(mdl)
    t0 = time.time()
    m.fit(Xtr_r, ytr_r)
    fit_time = time.time() - t0

    t0 = time.time()
    y_pred = m.predict(Xte_s)
    pred_time = time.time() - t0

    y_proba = m.predict_proba(Xte_s)[:, 1] if hasattr(m, 'predict_proba') else None
    return {
        'fit_time_s': round(fit_time, 4),
        'pred_time_ms': round(pred_time * 1000, 2),
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'auc': roc_auc_score(y_test, y_proba) if y_proba is not None else None,
    }

summary_rows = []
for _, row in best_per_model.iterrows():
    timing = time_model(MODELS[row['Model']], row['Scaler'], row['Resampler'])
    summary_rows.append({'Model': row['Model'], 'Variant': 'default',
                         'Scaler': row['Scaler'], 'Resampler': row['Resampler'],
                         **timing})

# Tambah W-KNN tuned
tuned_metrics = metrik(tuned_model)
summary_rows.append({
    'Model': 'W-KNN', 'Variant': 'tuned',
    'Scaler': BEST['Scaler'], 'Resampler': BEST['Resampler'],
    'fit_time_s': np.nan, 'pred_time_ms': np.nan,
    'accuracy': tuned_metrics['accuracy'],
    'precision': tuned_metrics['precision'],
    'recall': tuned_metrics['recall'],
    'f1': tuned_metrics['f1'],
    'auc': tuned_metrics['roc_auc'],
})

summary_df = pd.DataFrame(summary_rows).sort_values('auc', ascending=False).reset_index(drop=True)
print('=== Konsolidasi Performa Semua Kandidat (diurutkan ROC AUC) ===')
print(summary_df.round(4).to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels = [f"{r['Model']} ({r['Variant']})" for _, r in summary_df.iterrows()]
colors = []
for _, r in summary_df.iterrows():
    if 'W-KNN' in r['Model']:
        colors.append('#e67e22')   # oranye = base method
    else:
        colors.append('#3498db')

axes[0].barh(labels, summary_df['f1'], color=colors)
axes[0].set_xlabel('F1-Score (test set)')
axes[0].set_title('F1-Score Tiap Kandidat Model')
axes[0].set_xlim(0, 1.05)
for i, v in enumerate(summary_df['f1']):
    axes[0].text(v + 0.005, i, f"{v:.4f}", va='center')

axes[1].barh(labels, summary_df['auc'], color=colors)
axes[1].set_xlabel('ROC AUC (test set)')
axes[1].set_title('ROC AUC Tiap Kandidat Model')
axes[1].set_xlim(0, 1.05)
for i, v in enumerate(summary_df['auc']):
    axes[1].text(v + 0.005, i, f"{v:.4f}", va='center')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#e67e22', label='Base method (W-KNN)'),
    Patch(facecolor='#3498db', label='Kandidat lain'),
]
axes[1].legend(handles=legend_elements, loc='lower right', fontsize=8)

plt.tight_layout()
plt.show()


### 🎯 Kesimpulan

- **W-KNN (Weighted KNN)** dengan `MinMaxScaler` memberikan performa
  sangat baik (akurasi & ROC AUC tinggi) berkat dataset CKD yang
  separable di ruang Euclidean setelah scaling.
- Dataset CKD relatif kecil (400 pasien) — model non-parametric seperti
  W-KNN cocok karena tidak butuh asumsi distribusi.
- **Pelajaran preprocessing:**
  - Strip whitespace di kolom string krusial — tanpa ini target
    `'ckd'` dan `'ckd\t'` akan dianggap kelas berbeda.
  - Imputasi median/modus efektif untuk dataset dengan missing masif.
  - Scaler **MinMax** umumnya lebih baik dari Standard untuk W-KNN
    pada dataset dengan distribusi non-Gaussian.
